# Controlled synthetic forecasting — statistical baselines

Compare training-fitted joint Gaussian, shared-coefficient Gaussian AR(1), and causal EWMA Gaussian forecasts.
The objective is a fair comparison: a correctly specified statistical model should be competitive.
No diffusion training or neural evaluation is launched.

These small CPU pilots use the **first 64 validation targets**, 128 scenarios per target, seed 42,
and the unchanged Gaussian and mean-reverting configurations (12 assets, 60-day lookback, one-day horizon).
Train cutoff: 2009-02-27; validation cutoff: 2014-11-28. The test period stays held out.
All scoring uses observable arithmetic decimal returns; baseline estimation uses `log1p` of observations.

**Student-t is excluded:** exponentiated Student-t returns have no finite positive population moments.
Its separate mathematical diagnostics remain valid. Small finite-ensemble scores do not establish significance.

## Setup

Enable Kaggle Internet; CPU is sufficient. The established sync cell selects `feature/synthetic-market`.
No previously trained model or historical checkpoint is required.

In [ ]:
import os
import sys

REPO_PATH = "/kaggle/working/diffusion-portfolio"
USERNAME = "Amir-Entezari"
REPO = "diffusion-portfolio"
BRANCH = "feature/synthetic-market"

repo_url = f"https://github.com/{USERNAME}/{REPO}.git"

# Clone the specified branch if the repository doesn't exist.
if not os.path.exists(os.path.join(REPO_PATH, ".git")):
    !git clone --branch {BRANCH} --single-branch {repo_url} {REPO_PATH}

else:
    # Fetch latest changes from the specified branch.
    !git -C {REPO_PATH} fetch origin {BRANCH}

    # Switch to the branch and force-sync with remote.
    !git -C {REPO_PATH} checkout -B {BRANCH} FETCH_HEAD
    !git -C {REPO_PATH} reset --hard FETCH_HEAD

os.chdir(REPO_PATH)

SRC_PATH = os.path.join(REPO_PATH, "src")

if SRC_PATH not in sys.path:
    sys.path.insert(0, SRC_PATH)

# Verify the active branch and latest commit.
!git branch --show-current
!git log -1 --oneline

print(f"Repository synced with origin/{BRANCH}.")

In [ ]:
!pip install -q -e '.[dev]'
assert _exit_code == 0, "Dependency installation failed."

## Mathematical and causality tests

Tests check fitted moments, innovation covariance, stationary AR fitting, PSD EWMA updates, exact target alignment,
future-observation corruption, reproducible subsets, shared metric reuse and the Student-t guard.

In [ ]:
!pytest -q tests/unit/test_statistical_baselines.py tests/unit/test_synthetic_forecast_benchmarks.py tests/semantic/test_data_leakage.py
assert _exit_code == 0, "Statistical baseline tests failed."

## Gaussian market pilot

Fit once on 5,000 unique training observations. No fitting uses overlapping windows or validation targets.
EWMA starts from the training MLE and incorporates earlier validation observations only after they become known.
Its fixed decay is 0.94; it is not selected using held-out data.

Each output directory must be empty or absent. Use a new directory for another run; existing artifacts are never replaced.

In [ ]:
!PYTHONUNBUFFERED=1 python -u scripts/benchmark_synthetic_forecasts.py \
    --config configs/synthetic_gaussian.yaml \
    --models gaussian,ar1,ewma --split validation --n-scenarios 128 --limit 64 --seed 42 \
    --output-dir /kaggle/working/synthetic_forecast_benchmarks/gaussian_pilot
assert _exit_code == 0, "Gaussian benchmark failed."

## Mean-reverting return pilot

The model fits a common AR coefficient from observed training log returns. It does not receive the generator's
coefficient, means, factor loadings, regimes or covariance. Residual covariance describes one-step innovations,
not stationary marginal return covariance. A strong AR result here would be scientifically expected.

In [ ]:
!PYTHONUNBUFFERED=1 python -u scripts/benchmark_synthetic_forecasts.py \
    --config configs/synthetic_mean_reverting.yaml \
    --models gaussian,ar1,ewma --split validation --n-scenarios 128 --limit 64 --seed 42 \
    --output-dir /kaggle/working/synthetic_forecast_benchmarks/ar1_pilot
assert _exit_code == 0, "Mean-reverting benchmark failed."

## Saved validation results

Scores below come only from these executed pilots. Lower CRPS/Energy Score and smaller mean absolute calibration
error are better. Monte Carlo sampling and the short validation subset limit interpretation; no significance is claimed.
Each model uses identical target dates, assets, observations and ensemble size.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

OUT = Path("/kaggle/working/synthetic_forecast_benchmarks")
frames = []
for run in ("gaussian_pilot", "ar1_pilot"):
    table = pd.read_csv(OUT / run / "comparison.csv")
    table.insert(0, "dataset", json.loads((OUT / run / "report.json").read_text())["dataset"])
    frames.append(table)
display(pd.concat(frames, ignore_index=True))

In [ ]:
for run in ("gaussian_pilot", "ar1_pilot"):
    report = json.loads((OUT / run / "report.json").read_text())
    print(report["dataset"], "—", report["n_dates"], "validation dates")
    display(pd.DataFrame([
        {"model": name, **row, "absolute_error": abs(row["ace"])}
        for name, result in report["models"].items() for row in result["calibration"]
    ]))
    print("Fitted AR coefficient:", report["models"]["ar1"]["parameters"]["phi"])

## Illustrative forecast distributions

Show only the first validation date and first asset, chosen in advance. These examples are **not representative
performance evidence**. Observations are read from the same scenario archive used for the saved scores.

In [ ]:
for run in ("gaussian_pilot", "ar1_pilot"):
    fig, ax = plt.subplots(figsize=(8, 4))
    for name in ("gaussian", "ar1", "ewma"):
        with np.load(OUT / run / name / "validation_scenarios.npz", allow_pickle=False) as saved:
            ax.hist(saved["scenarios"][0, :, 0], bins=20, density=True, alpha=.35, label=name)
            observed = saved["observed"][0, 0]
            date, asset = str(saved["dates"][0]), str(saved["columns"][0])
    ax.axvline(observed, color="black", linestyle="--", label="Observed return")
    ax.set(title=f"Example only: {run} | {date} | {asset}", xlabel="Arithmetic decimal return", ylabel="Density")
    ax.legend()
    ax.grid(alpha=.2)
    plt.show()

## Artifacts and next comparisons

Each pilot saves resolved `config.yaml`, train-fitted `standardizer.npz`, `report.json`, `comparison.csv`,
and one `validation_scenarios.npz` / `validation_probabilistic_metrics.json` pair per baseline.
Archives use the existing `[date, scenario, asset]` format. Reports record source fingerprints, parameters,
NumPy version, seeds, dates, units and runtime. Save these isolated working directories with the Kaggle version.

No central artifact dataset is uploaded automatically. Diffusion comparison is deferred until exact source/config,
asset ordering, dates and observed arrays can be verified against saved diffusion forecasts. No test evaluation here.